# Notebook 24 — Boundary-aligned HSBD-v2 constraints

Notebook 18 improved calibration and fine macro-F1, but increased the actual
benign-to-attack rate because its dual used **mean attack probability on benign flows**,
not the logit-boundary event that creates a false alert.

This notebook compares:

1. CE + standard KD;
2. the original probability-proxy HSBD dual;
3. **HSBD-v2**, using binary logit-margin, ASVG edge-margin, family, and class-tail
   constraints.

All variants start from the same physically pruned 40%-FLOP structure and are selected
lexicographically: satisfy actual validation safety limits first, then minimise AWBIR/HSR,
then maximise family/fine macro-F1. No test data are used.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, platform, hashlib
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

candidates = [
    os.environ.get("SABER_REPO"),
    "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression",
    str(Path.cwd()),
]
REPO = None
for candidate in candidates:
    if not candidate:
        continue
    p = Path(candidate).expanduser()
    if (p / "src/saber").is_dir() and (p / "config").is_dir():
        REPO = p.resolve()
        break
if REPO is None:
    raise FileNotFoundError("Set SABER_REPO to the saber-ids-method repository.")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Repository:", REPO)
print("Device:", DEVICE)
print("Python:", sys.version.split()[0], "|", platform.platform())


In [ ]:
from copy import deepcopy
from torch import nn

from src.saber.bridge_ciciot import load_bridge
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate
from src.saber.adapters import collect_logits
from src.saber.surgery import prune_cnn1d_channels
from src.saber.losses import (
    HierarchyTensors, SaberLossConfig, saber_loss, edges_to_tensors,
)
from src.saber.constraints import differentiable_risk_proxies
from src.saber.constraints_v2 import (
    boundary_aligned_risk_proxies, directed_edge_violation_proxy,
)
from src.saber.postg5 import lexicographic_checkpoint_index

OUT = REPO / "results/saber/24_boundary_aligned_hsbd"
OUT.mkdir(parents=True, exist_ok=True)
TRAIN_LOADER, VAL_LOADER, TEST_LOADER, TEACHER, CLASS_NAMES = load_bridge()
TEACHER = TEACHER.to(DEVICE).eval()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES)
graph = pd.read_csv(REPO / "results/saber/14_risk_graph/asvg_edges_robust.csv")
edge_source, edge_target, edge_weight = edges_to_tensors(graph)
edge_source = edge_source.to(DEVICE)
edge_target = edge_target.to(DEVICE)
edge_weight = edge_weight.to(DEVICE)

hierarchy = HierarchyTensors(
    taxonomy.class_to_family_index,
    benign_index=taxonomy.benign_index,
    n_families=len(taxonomy.families),
).to(DEVICE)

# Prefer the new lookahead structure when it has passed; otherwise use the
# frozen V-C 40%-FLOP structure so the result remains comparable with NB18.
asl_gate = REPO / "results/saber/23_alert_semantic_lookahead/ASL_selection_gate.json"
if asl_gate.exists() and json.loads(asl_gate.read_text()).get("passed"):
    removed_path = REPO / "results/saber/23_alert_semantic_lookahead/shallow_selected_groups.csv"
    structure_name = "asl_select"
else:
    candidate = REPO / "results/saber/17b_calibrated_checkpoint_freeze/saber_v2_r40cal_removed_groups.csv"
    removed_path = candidate if candidate.exists() else (
        REPO / "results/saber/17_structured_selection/saber_v2_r40cal_removed_groups.csv"
    )
    structure_name = "saber_v2_static"

removed = pd.read_csv(removed_path)
prune_map = {
    layer: sorted(frame["channel_index"].astype(int).tolist())
    for layer, frame in removed.groupby("module_path")
}
example = next(iter(VAL_LOADER))[0][:8].to(DEVICE)
BASE_STUDENT, _ = prune_cnn1d_channels(
    TEACHER, prune_map, example, minimum_remaining_per_layer=4
)
BASE_STUDENT = BASE_STUDENT.to(DEVICE)
print("Starting structure:", structure_name, "| removed groups:", len(removed))


In [ ]:
@torch.no_grad()
def validation_logits(model):
    return collect_logits(model, VAL_LOADER, device=DEVICE)[:2]

teacher_val_logits, val_labels = validation_logits(TEACHER)
teacher_audit = full_model_audit(
    teacher_val_logits, val_labels, taxonomy, DEFAULT_COST_PROFILES
)

@torch.no_grad()
def actual_audit(model):
    logits, labels = validation_logits(model)
    audit = full_model_audit(logits, labels, taxonomy, DEFAULT_COST_PROFILES)
    awbir, _ = action_weighted_boundary_inversion_rate(
        teacher_val_logits, logits, labels, graph
    )
    audit["awbir"] = float(awbir)
    return audit

@torch.no_grad()
def average_proxy(model, proxy_kind):
    totals = {}
    count = 0
    edge_total = 0.0
    for x, y in VAL_LOADER:
        x, y = x.to(DEVICE), y.to(DEVICE)
        student = model(x)
        teacher = TEACHER(x)
        proxy = (
            differentiable_risk_proxies(student, y, hierarchy)
            if proxy_kind == "old"
            else boundary_aligned_risk_proxies(student, y, hierarchy)
        )
        batch_n = len(y)
        for key, value in proxy.items():
            totals[key] = totals.get(key, 0.0) + float(value.detach().cpu()) * batch_n
        edge_value = directed_edge_violation_proxy(
            student, teacher, y, edge_source, edge_target, edge_weight
        )
        edge_total += float(edge_value.detach().cpu()) * batch_n
        count += batch_n
    out = {key: value / max(count, 1) for key, value in totals.items()}
    out["edge_violation"] = edge_total / max(count, 1)
    return out

teacher_proxy_old = average_proxy(TEACHER, "old")
teacher_proxy_v2 = average_proxy(TEACHER, "v2")

def thresholds_from_teacher(values):
    return {
        key: float(value) * 1.10 + 1e-3
        for key, value in values.items()
    }
thresholds_old = thresholds_from_teacher(teacher_proxy_old)
thresholds_v2 = thresholds_from_teacher(teacher_proxy_v2)
print("Teacher actual:", teacher_audit)
print("v2 thresholds:", thresholds_v2)


In [ ]:
def train_variant(name, proxy_kind, use_hsbd, use_duals, epochs=8):
    torch.manual_seed(24026)
    np.random.seed(24026)
    model = deepcopy(BASE_STUDENT).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    config = (
        SaberLossConfig(
            fine_weight=1.0,
            family_weight=0.5,
            binary_weight=0.5,
            margin_weight=1.0,
            kd_weight=0.5,
            calibration_weight=0.1,
        )
        if use_hsbd else
        SaberLossConfig(
            fine_weight=1.0,
            family_weight=0.0,
            binary_weight=0.0,
            margin_weight=0.0,
            kd_weight=0.5,
            calibration_weight=0.0,
        )
    )
    thresholds = thresholds_old if proxy_kind == "old" else thresholds_v2
    lambdas = {key: 0.0 for key in thresholds}
    dual_step = 0.10
    max_dual = 50.0
    history = []
    states = []

    for epoch in range(1, epochs + 1):
        model.train()
        for x, y in TRAIN_LOADER:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            student_logits = model(x)
            with torch.no_grad():
                teacher_logits = TEACHER(x)

            proxy = (
                differentiable_risk_proxies(student_logits, y, hierarchy)
                if proxy_kind == "old"
                else boundary_aligned_risk_proxies(student_logits, y, hierarchy)
            )
            edge_proxy = directed_edge_violation_proxy(
                student_logits, teacher_logits, y,
                edge_source, edge_target, edge_weight,
            )
            proxy = {**proxy, "edge_violation": edge_proxy}
            penalty = student_logits.new_zeros(())
            if use_duals:
                for key, value in proxy.items():
                    threshold = thresholds[key]
                    penalty = penalty + lambdas[key] * torch.relu(
                        value / max(threshold, 1e-6) - 1.0
                    )
            loss, _ = saber_loss(
                student_logits, teacher_logits, y, hierarchy,
                edge_source, edge_target, edge_weight, config,
                extra_penalty=penalty,
            )
            loss.backward()
            optimizer.step()

        model.eval()
        actual = actual_audit(model)
        avg_proxy = average_proxy(model, proxy_kind)
        if use_duals:
            for key, value in avg_proxy.items():
                threshold = thresholds[key]
                normalized_violation = value / max(threshold, 1e-6) - 1.0
                lambdas[key] = float(np.clip(
                    lambdas[key] + dual_step * normalized_violation,
                    0.0, max_dual,
                ))
        row = {
            "variant": name,
            "epoch": epoch,
            **{f"val_{k}": float(v) for k, v in actual.items() if np.isscalar(v)},
            **{f"proxy_{k}": float(v) for k, v in avg_proxy.items()},
            **{f"dual_{k}": float(v) for k, v in lambdas.items()},
        }
        history.append(row)
        states.append(deepcopy(model.state_dict()))
        pd.DataFrame(history).to_csv(OUT / f"{name}_history.csv", index=False)
        print(name, epoch, "AWBIR=", round(actual["awbir"], 4),
              "B->A=", round(actual["benign_to_attack_rate"], 4),
              "ECE=", round(actual["ece15"], 4))

    history_df = pd.DataFrame(history)
    # Actual, not surrogate, feasibility.
    constraints = {
        "val_attack_to_benign_rate": (
            "max", teacher_audit["attack_to_benign_rate"] + 0.01
        ),
        "val_benign_to_attack_rate": (
            "max", teacher_audit["benign_to_attack_rate"] + 0.05
        ),
    }
    best_idx = lexicographic_checkpoint_index(
        history_df,
        constraints=constraints,
        primary_minimize=("val_awbir", "val_hsr_balanced_soc", "val_ece15"),
        secondary_maximize=("val_family_macro_f1", "val_fine_macro_f1"),
    )
    model.load_state_dict(states[best_idx])
    checkpoint = OUT / f"{name}_checkpoint.pt"
    torch.save({
        "state_dict": model.cpu().state_dict(),
        "variant": name,
        "structure": structure_name,
        "best_epoch": int(history_df.loc[best_idx, "epoch"]),
        "prune_map": prune_map,
        "class_names": CLASS_NAMES,
    }, checkpoint)
    model = model.to(DEVICE)
    final = actual_audit(model)
    return model, history_df, final, int(best_idx)

variants = [
    ("ce_kd", "v2", False, False),
    ("hsbd_old_dual", "old", True, True),
    ("hsbd_margin_dual", "v2", True, True),
]
summary = []
for name, proxy_kind, use_hsbd, use_duals in variants:
    _, history, final, best_idx = train_variant(
        name, proxy_kind, use_hsbd, use_duals
    )
    summary.append({
        "variant": name,
        "structure": structure_name,
        "best_epoch": int(history.loc[best_idx, "epoch"]),
        **{key: float(value) for key, value in final.items() if np.isscalar(value)},
    })

summary = pd.DataFrame(summary)
summary.to_csv(OUT / "hsbd_v2_summary.csv", index=False)
display(summary)


In [ ]:
ce = summary[summary["variant"] == "ce_kd"].iloc[0]
v2 = summary[summary["variant"] == "hsbd_margin_dual"].iloc[0]
gate = {
    "gate": "boundary_aligned_hsbd_v2",
    "passed": bool(
        v2["benign_to_attack_rate"] <= ce["benign_to_attack_rate"] + 0.01
        and v2["attack_to_benign_rate"] <= ce["attack_to_benign_rate"] + 0.01
        and v2["awbir"] <= ce["awbir"] + 0.01
        and v2["ece15"] <= 0.80 * ce["ece15"]
    ),
    "criteria": {
        "benign_to_attack_no_worse_than_ce_plus": 0.01,
        "attack_to_benign_no_worse_than_ce_plus": 0.01,
        "awbir_no_worse_than_ce_plus": 0.01,
        "ece_relative_max": 0.80,
    },
    "ce_kd": ce.to_dict(),
    "hsbd_margin_dual": v2.to_dict(),
}
(OUT / "HSBD_v2_gate.json").write_text(json.dumps(gate, indent=2), encoding="utf-8")
print(json.dumps(gate, indent=2))
